# 🌫️ Air Quality Prediction System
### Real-Time AQI Prediction using Historical Training + Live API Data

**Flow:**
```
User enters city → Fetch live pollutant data (AQICN API) → Model predicts AQI
→ Explain air quality → Give health suggestions → Display in Streamlit UI
```

---
**Dataset:** `city_day.csv` — 29,531 records across Indian cities  
**Features:** PM2.5, PM10, NO2, SO2, CO, O3  
**Target:** AQI  
**Models:** Linear Regression vs Random Forest Regressor (best selected automatically)


## ⚙️ STEP 1 — Install Dependencies

In [1]:
# Install all required packages
!pip install -q streamlit pyngrok joblib scikit-learn pandas numpy requests
print("✅ All packages installed successfully!")

✅ All packages installed successfully!



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 📦 STEP 2 — Import Libraries

In [2]:
import pandas as pd
import numpy as np
import joblib
import requests
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.impute import SimpleImputer

print("✅ All libraries imported!")

✅ All libraries imported!


## 📊 STEP 3 — Data Preprocessing

This function:
- Loads `city_day.csv`
- Selects only the 6 pollutant features + AQI
- Drops rows where AQI is missing (can't train without target)
- Imputes remaining missing pollutant values with column median
- Splits into train/test sets
- Scales features using StandardScaler

In [3]:
FEATURES = ['PM2.5', 'PM10', 'NO2', 'SO2', 'CO', 'O3']
TARGET   = 'AQI'

def preprocess_data(filepath='city_day.csv'):
    """
    Load and clean the dataset.

    Returns:
        X_train, X_test, y_train, y_test  — scaled numpy arrays
        scaler                            — fitted StandardScaler (needed for inference)
    """
    # --- 1. Load ---
    df = pd.read_csv(filepath)
    print(f"📂 Loaded dataset: {df.shape[0]} rows × {df.shape[1]} columns")

    # --- 2. Select relevant columns ---
    df = df[FEATURES + [TARGET]].copy()

    # --- 3. Drop rows where AQI (target) is missing ---
    before = len(df)
    df.dropna(subset=[TARGET], inplace=True)
    print(f"🗑️  Dropped {before - len(df)} rows with missing AQI → {len(df)} rows remain")

    # --- 4. Impute missing pollutant values with column median ---
    imputer = SimpleImputer(strategy='median')
    df[FEATURES] = imputer.fit_transform(df[FEATURES])
    print(f"🔧 Missing pollutant values imputed with column medians")

    # --- 5. Split features / target ---
    X = df[FEATURES].values
    y = df[TARGET].values

    # --- 6. Train/test split (80/20) ---
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # --- 7. Scale features ---
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test  = scaler.transform(X_test)

    print(f"✅ Train size: {X_train.shape[0]} | Test size: {X_test.shape[0]}")
    return X_train, X_test, y_train, y_test, scaler


# Run preprocessing
X_train, X_test, y_train, y_test, scaler = preprocess_data('city_day.csv')

📂 Loaded dataset: 29531 rows × 16 columns
🗑️  Dropped 4681 rows with missing AQI → 24850 rows remain
🔧 Missing pollutant values imputed with column medians
✅ Train size: 19880 | Test size: 4970


## 🤖 STEP 4 — Model Training & Comparison

We train both:
- **Linear Regression** — simple, interpretable baseline
- **Random Forest Regressor** — more powerful, handles non-linearity

The model with lower RMSE (and higher R²) is automatically selected.

In [4]:
def train_model(X_train, X_test, y_train, y_test):
    """
    Train Linear Regression and Random Forest models.
    Compare on RMSE and R² Score.
    Return the best model.
    """
    candidates = {
        'Linear Regression':    LinearRegression(),
        'Random Forest':        RandomForestRegressor(
                                    n_estimators=100,
                                    max_depth=15,
                                    random_state=42,
                                    n_jobs=-1
                                )
    }

    results = {}
    trained_models = {}

    print("\n🏋️  Training models...\n")
    print(f"{'Model':<25} {'RMSE':>10} {'R² Score':>12}")
    print("-" * 50)

    for name, model in candidates.items():
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        rmse = np.sqrt(mean_squared_error(y_test, preds))
        r2   = r2_score(y_test, preds)

        results[name] = {'rmse': rmse, 'r2': r2}
        trained_models[name] = model

        print(f"{name:<25} {rmse:>10.2f} {r2:>12.4f}")

    # Pick the model with the lowest RMSE
    best_name = min(results, key=lambda k: results[k]['rmse'])
    best_model = trained_models[best_name]

    print(f"\n🏆 Best Model: {best_name}")
    print(f"   → RMSE   : {results[best_name]['rmse']:.2f}")
    print(f"   → R² Score: {results[best_name]['r2']:.4f}")

    return best_model, best_name


best_model, best_name = train_model(X_train, X_test, y_train, y_test)


🏋️  Training models...

Model                           RMSE     R² Score
--------------------------------------------------
Linear Regression              59.53       0.8065
Random Forest                  41.70       0.9050

🏆 Best Model: Random Forest
   → RMSE   : 41.70
   → R² Score: 0.9050


## 💾 STEP 5 — Save & Load Model

In [5]:
MODEL_PATH  = 'aqi_model.pkl'
SCALER_PATH = 'aqi_scaler.pkl'

def save_model(model, scaler, model_path=MODEL_PATH, scaler_path=SCALER_PATH):
    """Persist model and scaler to disk using joblib."""
    joblib.dump(model,  model_path)
    joblib.dump(scaler, scaler_path)
    print(f"✅ Model  saved → {model_path}")
    print(f"✅ Scaler saved → {scaler_path}")


def load_model(model_path=MODEL_PATH, scaler_path=SCALER_PATH):
    """Load model and scaler from disk."""
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model file not found: {model_path}")
    if not os.path.exists(scaler_path):
        raise FileNotFoundError(f"Scaler file not found: {scaler_path}")

    model  = joblib.load(model_path)
    scaler = joblib.load(scaler_path)
    print(f"✅ Model and scaler loaded from disk.")
    return model, scaler


# Save the best model
save_model(best_model, scaler)

# Verify round-trip
loaded_model, loaded_scaler = load_model()
print(f"   Type: {type(loaded_model).__name__}")

✅ Model  saved → aqi_model.pkl
✅ Scaler saved → aqi_scaler.pkl
✅ Model and scaler loaded from disk.
   Type: RandomForestRegressor


## 🌐 STEP 6 — AQICN API Integration

We use the **World Air Quality Index (AQICN)** API to fetch real-time pollutant data.

**Get a FREE API token:** https://aqicn.org/api/

The API returns `iaqi` (individual AQI indices) which map directly to the pollutants we trained on.

In [ ]:
# ============================================================
#  ⚠️  ENTER YOUR FREE AQICN API TOKEN BELOW
#  Get it at: https://aqicn.org/api/
# ============================================================
# import os
# AQICN_API_TOKEN = os.getenv("AQICN_API_KEY")  # <-- Replace this

AQICN_API_KEY='f1a84cec8ce821803f922fc4d119f07ee16d6bd7'
# ============================================================

print("🔑 API token configured. Replace 'YOUR_AQICN_TOKEN_HERE' with your real token.")

🔑 API token configured. Replace 'YOUR_AQICN_TOKEN_HERE' with your real token.


In [7]:
def fetch_pollution_data(city: str, token: str = AQICN_API_TOKEN) -> dict:
    """
    Fetch real-time pollutant data for a given city from AQICN API.

    Args:
        city  : City name (e.g., 'Delhi', 'Mumbai')
        token : AQICN API token

    Returns:
        dict with keys: PM2.5, PM10, NO2, SO2, CO, O3
        Returns None if city not found or API fails.
    """
    # Sanitise city name for URL
    city_slug = city.strip().lower().replace(' ', '-')
    url = f"https://api.waqi.info/feed/{city_slug}/?token={token}"

    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()   # raises HTTPError on 4xx/5xx

        data = response.json()

        # API returns status 'error' for invalid city/token
        if data.get('status') != 'ok':
            msg = data.get('data', 'Unknown error from AQICN API')
            raise ValueError(f"AQICN API error: {msg}")

        iaqi = data['data'].get('iaqi', {})

        # Map AQICN keys → our feature names
        #  AQICN uses: pm25, pm10, no2, so2, co, o3
        pollutants = {
            'PM2.5': iaqi.get('pm25', {}).get('v', np.nan),
            'PM10':  iaqi.get('pm10', {}).get('v', np.nan),
            'NO2':   iaqi.get('no2',  {}).get('v', np.nan),
            'SO2':   iaqi.get('so2',  {}).get('v', np.nan),
            'CO':    iaqi.get('co',   {}).get('v', np.nan),
            'O3':    iaqi.get('o3',   {}).get('v', np.nan),
        }

        # Count how many pollutants were actually available
        available = sum(1 for v in pollutants.values() if not np.isnan(v))
        print(f"🌐 Fetched data for '{city}': {available}/6 pollutants available")
        print(f"   {pollutants}")

        return pollutants

    except requests.exceptions.Timeout:
        raise ConnectionError("⏰ API request timed out. Check your internet connection.")
    except requests.exceptions.ConnectionError:
        raise ConnectionError("📡 Cannot reach AQICN API. Check your internet connection.")
    except requests.exceptions.HTTPError as e:
        raise ConnectionError(f"🔴 HTTP error: {e}")


# ---- Quick test (uses demo token for illustration) ----
# Uncomment the lines below once you have a real token:
# data = fetch_pollution_data('Delhi')
# print(data)
print("✅ fetch_pollution_data() defined. Add your token above and test with any city.")

✅ fetch_pollution_data() defined. Add your token above and test with any city.


## 🔮 STEP 7 — Prediction Pipeline, AQI Category & Health Advice

In [8]:
def predict_aqi(pollutant_dict: dict, model, scaler) -> float:
    """
    Format pollutant data and predict AQI using the trained model.

    Args:
        pollutant_dict : {'PM2.5': v, 'PM10': v, 'NO2': v, 'SO2': v, 'CO': v, 'O3': v}
        model          : trained sklearn model
        scaler         : fitted StandardScaler

    Returns:
        Predicted AQI (float, clipped to minimum 0)
    """
    # Build a 1-row DataFrame in the same column order as training
    row = pd.DataFrame([pollutant_dict], columns=FEATURES)

    # Impute any NaN values in live data using column medians from training
    # (If a sensor doesn't report a pollutant, we use its median as fallback)
    # We use a simple fill from a known dataset-level median for safety
    # Alternatively: imputer could be saved alongside the model
    DATASET_MEDIANS = {
        'PM2.5': 58.0, 'PM10': 90.0, 'NO2': 22.0,
        'SO2': 11.0,   'CO':   1.0,  'O3':  31.0
    }
    for col in FEATURES:
        if np.isnan(row[col].values[0]):
            row[col] = DATASET_MEDIANS[col]

    # Scale using the same scaler fitted on training data
    X_scaled = scaler.transform(row.values)

    # Predict
    aqi_pred = float(model.predict(X_scaled)[0])
    aqi_pred = max(0.0, round(aqi_pred, 1))   # AQI can't be negative
    return aqi_pred


def get_aqi_category(aqi: float) -> dict:
    """
    Classify AQI into a named category and return display metadata.

    Returns a dict with: category, color, emoji
    """
    if aqi <= 50:
        return {'category': 'Good',      'color': '#00e400', 'emoji': '🟢'}
    elif aqi <= 100:
        return {'category': 'Moderate',  'color': '#ffff00', 'emoji': '🟡'}
    elif aqi <= 200:
        return {'category': 'Poor',      'color': '#ff7e00', 'emoji': '🟠'}
    elif aqi <= 300:
        return {'category': 'Very Poor', 'color': '#ff0000', 'emoji': '🔴'}
    else:
        return {'category': 'Severe',    'color': '#7e0023', 'emoji': '🟣'}


def get_health_advice(aqi: float) -> list[str]:
    """
    Return a list of health advice strings based on AQI level.
    """
    if aqi <= 50:
        return [
            "✅ Air quality is good. Enjoy outdoor activities!",
            "✅ Safe for all age groups including children and elderly.",
            "✅ No special precautions needed."
        ]
    elif aqi <= 100:
        return [
            "⚠️  Acceptable air quality, but sensitive individuals should be cautious.",
            "⚠️  People with asthma or respiratory issues — limit prolonged outdoor exertion.",
            "✅ Healthy individuals can continue normal activity."
        ]
    elif aqi <= 200:
        return [
            "🚫 Air quality is Poor. Limit outdoor activities.",
            "😷 Wear an N95 mask if going outdoors.",
            "🏠 Children, elderly, and those with heart/lung disease should stay indoors.",
            "🪟 Keep windows closed. Use an air purifier indoors."
        ]
    elif aqi <= 300:
        return [
            "🚨 Air quality is Very Poor! Avoid all outdoor activities.",
            "😷 Always wear an N95 mask if you must go outside.",
            "🏠 Everyone — especially vulnerable groups — should stay indoors.",
            "🪟 Keep all windows and doors sealed. Run air purifiers on high.",
            "💧 Stay hydrated. Avoid physical exertion."
        ]
    else:
        return [
            "🆘 SEVERE air pollution! Health emergency conditions.",
            "🚫 Do NOT go outside under any circumstances.",
            "😷 If going out is unavoidable, use an N95/N99 respirator.",
            "🏥 Seek medical attention immediately if you experience breathing difficulty.",
            "📴 Close all ventilation. Use air purifiers continuously.",
            "📲 Follow local authority advisories."
        ]


print("✅ Prediction pipeline, AQI categoriser, and health advisor defined!")

# ---- Quick offline test ----
test_data = {
    'PM2.5': 85.0, 'PM10': 120.0, 'NO2': 40.0,
    'SO2': 15.0,   'CO':   1.2,   'O3':  28.0
}
aqi_val  = predict_aqi(test_data, loaded_model, loaded_scaler)
cat_info = get_aqi_category(aqi_val)
advice   = get_health_advice(aqi_val)

print(f"\n🧪 Test Prediction:")
print(f"   Predicted AQI  : {aqi_val}")
print(f"   Category       : {cat_info['emoji']} {cat_info['category']}")
print(f"   Health Advice  :")
for tip in advice:
    print(f"     {tip}")

✅ Prediction pipeline, AQI categoriser, and health advisor defined!

🧪 Test Prediction:
   Predicted AQI  : 185.5
   Category       : 🟠 Poor
   Health Advice  :
     🚫 Air quality is Poor. Limit outdoor activities.
     😷 Wear an N95 mask if going outdoors.
     🏠 Children, elderly, and those with heart/lung disease should stay indoors.
     🪟 Keep windows closed. Use an air purifier indoors.


## 🎨 STEP 8 — Write the Streamlit App

We write the full Streamlit frontend to a file called `app.py`.
It is then launched using **pyngrok** to expose it as a public URL inside Colab.

In [9]:
app_code = '''
import streamlit as st
import numpy as np
import pandas as pd
import joblib
import requests
import os

# ============================================================
#  CONFIGURATION — update your AQICN token here too!
# ============================================================
AQICN_API_TOKEN = os.environ.get("AQICN_TOKEN", "YOUR_AQICN_TOKEN_HERE")
FEATURES = ["PM2.5", "PM10", "NO2", "SO2", "CO", "O3"]
DATASET_MEDIANS = {
    "PM2.5": 58.0, "PM10": 90.0, "NO2": 22.0,
    "SO2": 11.0,   "CO":   1.0,  "O3":  31.0
}

# ---- Load model & scaler ----
@st.cache_resource
def load_assets():
    model  = joblib.load("aqi_model.pkl")
    scaler = joblib.load("aqi_scaler.pkl")
    return model, scaler

model, scaler = load_assets()

# ---- Helper functions ----
def fetch_pollution_data(city):
    city_slug = city.strip().lower().replace(" ", "-")
    url = f"https://api.waqi.info/feed/{city_slug}/?token={AQICN_API_TOKEN}"
    try:
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()
        data = resp.json()
        if data.get("status") != "ok":
            return None, f"City not found or API error: {data.get(\"data\", \"Unknown\")}"
        iaqi = data["data"].get("iaqi", {})
        pollutants = {
            "PM2.5": iaqi.get("pm25", {}).get("v", np.nan),
            "PM10":  iaqi.get("pm10", {}).get("v", np.nan),
            "NO2":   iaqi.get("no2",  {}).get("v", np.nan),
            "SO2":   iaqi.get("so2",  {}).get("v", np.nan),
            "CO":    iaqi.get("co",   {}).get("v", np.nan),
            "O3":    iaqi.get("o3",   {}).get("v", np.nan),
        }
        return pollutants, None
    except Exception as e:
        return None, str(e)


def predict_aqi(pollutant_dict):
    row = pd.DataFrame([pollutant_dict], columns=FEATURES)
    for col in FEATURES:
        if np.isnan(row[col].values[0]):
            row[col] = DATASET_MEDIANS[col]
    X_scaled = scaler.transform(row.values)
    aqi = float(model.predict(X_scaled)[0])
    return max(0.0, round(aqi, 1))


def get_aqi_category(aqi):
    if aqi <= 50:   return ("Good",      "#00c853", "🟢")
    elif aqi <= 100: return ("Moderate",  "#ffd600", "🟡")
    elif aqi <= 200: return ("Poor",      "#ff6d00", "🟠")
    elif aqi <= 300: return ("Very Poor", "#dd2c00", "🔴")
    else:            return ("Severe",    "#6a0080", "🟣")


def get_health_advice(aqi):
    if aqi <= 50:
        return [
            "✅ Air quality is **Good**. Enjoy outdoor activities!",
            "✅ Safe for all age groups including children and elderly.",
        ]
    elif aqi <= 100:
        return [
            "⚠️ Acceptable quality, but sensitive individuals should be cautious.",
            "⚠️ People with asthma or respiratory issues — limit prolonged outdoor exertion.",
        ]
    elif aqi <= 200:
        return [
            "🚫 Limit outdoor activities.",
            "😷 Wear an **N95 mask** if going outdoors.",
            "🏠 Children, elderly, and those with heart/lung disease should stay indoors.",
            "🪟 Keep windows closed and use an air purifier.",
        ]
    elif aqi <= 300:
        return [
            "🚨 Avoid **all** outdoor activities!",
            "😷 Always wear an N95 mask outdoors.",
            "🏠 Everyone should stay indoors.",
            "🪟 Keep windows sealed. Run air purifiers on high.",
        ]
    else:
        return [
            "🆘 **SEVERE** pollution! Health emergency.",
            "🚫 **Do NOT go outside** under any circumstances.",
            "🏥 Seek medical attention if breathing is difficult.",
            "📴 Seal all ventilation. Use air purifiers continuously.",
        ]

# ============================================================
#  STREAMLIT UI
# ============================================================
st.set_page_config(
    page_title="🌫️ AQI Predictor",
    page_icon="🌫️",
    layout="centered"
)

# Custom CSS for clean design
st.markdown("""
<style>
    .main { background-color: #0e1117; }
    .aqi-box {
        border-radius: 16px;
        padding: 28px 36px;
        margin: 16px 0;
        text-align: center;
        border: 2px solid;
    }
    .aqi-number { font-size: 72px; font-weight: 900; line-height: 1.1; }
    .aqi-label  { font-size: 28px; font-weight: 700; margin-top: 4px; }
    .pollutant-card {
        background: #1e2130;
        border-radius: 12px;
        padding: 12px 16px;
        text-align: center;
    }
</style>
""", unsafe_allow_html=True)


st.title("🌫️ Real-Time AQI Prediction")
st.caption("Trained on Indian city_day.csv dataset · Powered by AQICN live API")
st.divider()

city_input = st.text_input(
    label="🔍 Enter City Name",
    placeholder="e.g. Delhi, Mumbai, Chennai, Kolkata...",
    help="Enter any city supported by the AQICN network."
)

check_btn = st.button("🔬 Check AQI", type="primary", use_container_width=True)

if check_btn:
    if not city_input.strip():
        st.warning("Please enter a city name first.")
    elif AQICN_API_TOKEN == "YOUR_AQICN_TOKEN_HERE":
        st.error("⛔ Please update AQICN_API_TOKEN in the app code with your real token.")
        st.info("Get a free token at: https://aqicn.org/api/")
    else:
        with st.spinner(f"🌐 Fetching live pollutant data for **{city_input}**..."):
            pollutants, error = fetch_pollution_data(city_input)

        if error:
            st.error(f"❌ Could not fetch data: {error}")
            st.info("Tips: Check the city name spelling, or try a nearby major city.")
        else:
            aqi_val   = predict_aqi(pollutants)
            cat, color, emoji = get_aqi_category(aqi_val)
            advice    = get_health_advice(aqi_val)

            # ---- AQI Result Box ----
            st.markdown(f"""
            <div class="aqi-box" style="border-color:{color}; background:{color}18">
                <div style="color: #aaa; font-size:16px; margin-bottom:8px;">📍 {city_input.title()}</div>
                <div class="aqi-number" style="color:{color}">{aqi_val}</div>
                <div class="aqi-label"  style="color:{color}">{emoji} {cat}</div>
            </div>
            """, unsafe_allow_html=True)

            # ---- Live Pollutant Values ----
            st.subheader("📊 Live Pollutant Levels")
            units = {
                "PM2.5": "µg/m³", "PM10": "µg/m³",
                "NO2":   "ppb",   "SO2":  "ppb",
                "CO":    "ppm",   "O3":   "ppb"
            }
            cols = st.columns(3)
            for i, feat in enumerate(FEATURES):
                val = pollutants.get(feat, None)
                display = f"{val:.1f} {units[feat]}" if val is not None and not np.isnan(val) else "N/A (imputed)"
                with cols[i % 3]:
                    st.markdown(f"""
                    <div class="pollutant-card">
                        <div style="font-size:12px; color:#888;">{feat}</div>
                        <div style="font-size:20px; font-weight:700; color:#fff;">{display}</div>
                    </div>
                    """, unsafe_allow_html=True)

            st.divider()

            # ---- Health Advice ----
            st.subheader(f"{emoji} Health Advice — {cat}")
            for tip in advice:
                st.markdown(tip)

            # ---- AQI Scale Reference ----
            with st.expander("📖 AQI Scale Reference"):
                scale_data = {
                    "Range": ["0–50", "51–100", "101–200", "201–300", "301+"],
                    "Category": ["Good", "Moderate", "Poor", "Very Poor", "Severe"],
                    "Color": ["🟢", "🟡", "🟠", "🔴", "🟣"]
                }
                st.table(pd.DataFrame(scale_data))

st.divider()
st.caption("Model trained on city_day.csv (India) · AQICN API for live data · sklearn RandomForest/LinearRegression")
'''

with open('app.py', 'w', encoding='utf-8') as f:
    f.write(app_code)

print("✅ Streamlit app written to app.py")

✅ Streamlit app written to app.py


## 🚀 STEP 9 — Launch Streamlit App via ngrok

Since Google Colab doesn't have a browser, we use **pyngrok** to create a public tunnel to the local Streamlit server.

**Before running:**
1. Sign up at https://ngrok.com and copy your auth token
2. Replace `YOUR_NGROK_TOKEN_HERE` below

In [10]:
import subprocess
import time
from pyngrok import ngrok, conf

# ============================================================
#  ⚠️  ENTER YOUR FREE NGROK AUTH TOKEN BELOW
#  Get it at: https://dashboard.ngrok.com/get-started/your-authtoken
# ============================================================
NGROK_AUTH_TOKEN = '3CXQyd0AbPYIMZ0dk5RI0zYUTjs_7mjNJsbXMEjzL59y3HcbQ'   # <-- Replace this
# ============================================================

# Also pass the AQICN token as environment variable to the app
import os
os.environ['AQICN_TOKEN'] = AQICN_API_TOKEN

# Set ngrok auth token
conf.get_default().auth_token = NGROK_AUTH_TOKEN

# Kill any existing Streamlit processes
!pkill -f streamlit 2>/dev/null || true

# Start Streamlit in background
proc = subprocess.Popen(
    ['streamlit', 'run', 'app.py',
     '--server.port=8501',
     '--server.headless=true',
     '--server.enableCORS=false'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Give Streamlit a moment to start
time.sleep(4)

# Open ngrok tunnel
tunnel = ngrok.connect(8501, 'http')

print("\n" + "="*60)
print("🚀  YOUR AQI APP IS LIVE!")
print("="*60)
print(f"\n👉  Open this URL in your browser:\n")
print(f"    {tunnel.public_url}")
print("\n" + "="*60)
print("💡  The app will stay live as long as this cell is running.")
print("    To stop, interrupt the kernel or run: ngrok.kill()")

The system cannot find the path specified.



🚀  YOUR AQI APP IS LIVE!

👉  Open this URL in your browser:

    https://attach-shelf-scariness.ngrok-free.dev

💡  The app will stay live as long as this cell is running.
    To stop, interrupt the kernel or run: ngrok.kill()


## 🧪 STEP 10 — Offline Test (No API Token Needed)

Test the complete prediction pipeline locally without any API calls.

In [11]:
def run_offline_demo(city_name: str, pollutants: dict):
    """
    Simulate the full pipeline with manually provided pollutant values.
    Useful for testing without a live API token.
    """
    print("=" * 55)
    print(f" 🌫️  AQI PREDICTION REPORT — {city_name.upper()}")
    print("=" * 55)

    # Show input
    print("\n📊 Pollutant Input Values:")
    for k, v in pollutants.items():
        print(f"   {k:<8}: {v}")

    # Predict
    aqi_val = predict_aqi(pollutants, loaded_model, loaded_scaler)
    cat_info = get_aqi_category(aqi_val)
    advice   = get_health_advice(aqi_val)

    # Display result
    print(f"\n🔮 Predicted AQI  : {aqi_val}")
    print(f"   Category       : {cat_info['emoji']} {cat_info['category']}")
    print("\n💊 Health Advice:")
    for tip in advice:
        print(f"   {tip}")
    print("=" * 55)


# Test Case 1 — Clean city
run_offline_demo("Clean City", {
    'PM2.5': 15.0, 'PM10': 30.0, 'NO2': 10.0,
    'SO2': 5.0,    'CO':   0.5,  'O3':  20.0
})

print()

# Test Case 2 — Heavily polluted city
run_offline_demo("Polluted City", {
    'PM2.5': 250.0, 'PM10': 350.0, 'NO2': 90.0,
    'SO2': 60.0,    'CO':   5.0,   'O3':  80.0
})

 🌫️  AQI PREDICTION REPORT — CLEAN CITY

📊 Pollutant Input Values:
   PM2.5   : 15.0
   PM10    : 30.0
   NO2     : 10.0
   SO2     : 5.0
   CO      : 0.5
   O3      : 20.0

🔮 Predicted AQI  : 44.0
   Category       : 🟢 Good

💊 Health Advice:
   ✅ Air quality is good. Enjoy outdoor activities!
   ✅ Safe for all age groups including children and elderly.
   ✅ No special precautions needed.

 🌫️  AQI PREDICTION REPORT — POLLUTED CITY

📊 Pollutant Input Values:
   PM2.5   : 250.0
   PM10    : 350.0
   NO2     : 90.0
   SO2     : 60.0
   CO      : 5.0
   O3      : 80.0

🔮 Predicted AQI  : 404.0
   Category       : 🟣 Severe

💊 Health Advice:
   🆘 SEVERE air pollution! Health emergency conditions.
   🚫 Do NOT go outside under any circumstances.
   😷 If going out is unavoidable, use an N95/N99 respirator.
   🏥 Seek medical attention immediately if you experience breathing difficulty.
   📴 Close all ventilation. Use air purifiers continuously.
   📲 Follow local authority advisories.


---
## ✅ Summary

| Component | Details |
|---|---|
| **Dataset** | `city_day.csv` — 29,531 records, 6 pollutants + AQI |
| **Preprocessing** | Median imputation, StandardScaler, 80/20 split |
| **Models** | Linear Regression vs Random Forest (best auto-selected) |
| **Evaluation** | RMSE + R² Score on test set |
| **Saved** | `aqi_model.pkl` + `aqi_scaler.pkl` via joblib |
| **API** | AQICN `waqi.info` — fetches PM2.5, PM10, NO2, SO2, CO, O3 |
| **Pipeline** | impute → scale → predict → categorise → advise |
| **Frontend** | Streamlit app with live pollutant grid + color-coded AQI box |
| **Hosting** | Exposed via ngrok public URL in Google Colab |

### 📁 Files Generated
- `aqi_model.pkl`  — trained prediction model
- `aqi_scaler.pkl` — StandardScaler for feature normalisation
- `app.py`         — Streamlit frontend application

### 🔑 Tokens You Need (Both Free)
- **AQICN API token** → https://aqicn.org/api/
- **ngrok auth token** → https://dashboard.ngrok.com/get-started/your-authtoken
